Dependencies

In [ ]:
import pandas as pd
import numpy as np
import statistics as sts
import matplotlib as plt
import seaborn as sns
rng = np.random.default_rng(12345)

I have also made use of the datawrangler extension to visualize the dataframes and get an idea of general missingness/unique id and whether ranges of numeric values are plausible. I have tried to also incorporate it into the code here where possible but I wanted to inform you of this in the Dependencies.

Loading the data

In [144]:
dfGenes = pd.read_csv("../data/genes.csv")
dfNL = pd.read_csv("../data/NL.csv")
dfPL = pd.read_csv("../data/PL.csv")
dfUK = pd.read_csv("../data/UK.csv")
dfUS = pd.read_csv("../data/US.csv")

In [ ]:
dfNL
dfNL["height"] = dfNL["height"].astype("Int64")
dfNL["weight"] = dfNL["weight"].astype("Int64")

,id,weight,height,sex,smokes
0,NL_0001,105,188,male,yes
1,NL_0002,114,177,male,yes
2,NL_0003,70,176,female,no
3,NL_0004,68,175,male,no
4,NL_0005,86,186,male,yes
...,...,...,...,...,...
1317,NL_1318,48,168,female,no
1318,NL_1319,97,181,male,no
1319,NL_1320,96,185,male,no
1320,NL_1321,77,176,female,no


 This is the standard to which the others must match, however since the other dataframes have some missing values in some columns, the dtype of the NL dataframe for height and weight of int64 will not work since I cannot encode missing values in those. As such I updated it to Int64 for merging later.

Checking and cleaning the dataset for PL

In [ ]:
dfPL # So missing values, misnamed height column, smoke status values differ and decimals in numeric variables, order is also wrong of columns. Lets fix it!

,id,weight,height,sex,smokes,BMI
0,S0001,77,178,female,no,24.302487
1,S0002,74,155,male,no,30.801249
2,S0003,70,173,male,yes,23.388687
3,S0004,95,167,male,no,34.063609
4,S0005,78,181,female,no,23.808797
...,...,...,...,...,...,...
1334,S0537,56,158,male,no,22.432303
1335,S0538,43,159,male,no,17.008821
1336,S0539,80,162,male,no,30.483158
1337,S053a,110,194,NaN,yes,29.227336


In [ ]:
dfPL["id"].is_unique # There are some ids that are not unique
dupes = dfPL[dfPL["id"].duplicated()] # These are the duplicated ids, its id "S030f" duplicated 199 times from row 783 to 981
dfPL = dfPL.drop(dupes.index).reset_index(drop=True) # Run once to drop these duplicated rows whilst reseting index




,id,heigth,weight,sex,smokes
0,S0001,178.0,77.0,female,False
1,S0002,155.0,74.0,male,False
2,S0003,173.0,70.0,male,True
3,S0004,167.0,95.0,male,False
4,S0005,181.0,78.0,female,False
...,...,...,...,...,...
1334,S0537,158.0,56.0,male,False
1335,S0538,159.0,43.0,male,False
1336,S0539,162.0,80.0,male,False
1337,S053a,194.0,110.0,NaN,True


In [ ]:
dfPL.loc[(dfPL.smokes == False), "smokes"] = "no" # Convert the smoking variable to the dutch standard
dfPL.loc[(dfPL.smokes == True), "smokes"] = "yes"

dfPL["smokes"] = dfPL["smokes"].astype(str) # Just so all dataframes have the same object type for each column

In [151]:
dfPL = dfPL.rename(columns={"heigth": "height"}) # Rename the height column to match dutch standard

In [ ]:
#dfPL.height.__len__() == dfPL.height[dfPL.height.notna()].__len__() # Missing values in height
#dfPL.weight.__len__() == dfPL.weight[dfPL.weight.notna()].__len__() # Missing values in weight
#dfPL.sex.__len__() == dfPL.sex[dfPL.sex.notna()].__len__()          # Missing values in sex
#dfPL.smokes.__len__() == dfPL.smokes[dfPL.smokes.notna()].__len__() # Missing values in smoking status

False

In [152]:
dfPL["height"] = dfPL["height"].round(0).astype("Int64") # Getting rid of decimals in the numeric variables to match dutch standard, had to use round(0) and astype("Int64") instead of just int because of missing values
dfPL["weight"] = dfPL["weight"].round(0).astype("Int64")

In [153]:
dfPL = dfPL.loc[:, ["id", "weight", "height", "sex", "smokes"]] # Reordering the columns to match dutch standard

In [ ]:
dfPL["BMI"] = (dfPL.weight)/(dfPL.height/100)**2 # Calculating BMI for polish data and adding to DF
dfPL # Looks good!

,id,weight,height,sex,smokes,BMI
0,S0001,77,178,female,no,24.302487
1,S0002,74,155,male,no,30.801249
2,S0003,70,173,male,yes,23.388687
3,S0004,95,167,male,no,34.063609
4,S0005,78,181,female,no,23.808797
...,...,...,...,...,...,...
1334,S0537,56,158,male,no,22.432303
1335,S0538,43,159,male,no,17.008821
1336,S0539,80,162,male,no,30.483158
1337,S053a,110,194,NaN,yes,29.227336


Checking and cleaning the data for the US dataset

In [ ]:
dfUS["id"].is_unique # US data ids are unique
dfUS # Only issue seems to be missing values which I elected to keep in the data for now instead of deleting rows with missing value. Ill turn height into whole number again:


In [ ]:
dfUS["height"] = dfUS["height"].round(0).astype("Int64") # Same thing again to get rid of decimals
dfUS["weight"] = dfUS["weight"].round(0).astype("Int64") # Couldn't see it in weight initially but no harm applying it anyways just in case
dfUS["BMI"] = (dfUS.weight)/(dfUS.height/100)**2 # Calculating BMI for US and adding to dataframe
dfUS # Looks good!

Checking and cleaning the data for the UK dataset

In [ ]:
dfUK["id"].is_unique # Unique ids.
dfUK

,id,weight,height,sex,smokes
0,NL_0001,105,188,male,yes
1,NL_0002,114,177,male,yes
2,NL_0003,70,176,female,no
3,NL_0004,68,175,male,no
4,NL_0005,86,186,male,yes
...,...,...,...,...,...
1317,NL_1318,48,168,female,no
1318,NL_1319,97,181,male,no
1319,NL_1320,96,185,male,no
1320,NL_1321,77,176,female,no


For the UK: Weight seems quite high in general, also in datawrangler so possible that this is in pounds instead of kilogram. Height is most certainly in inches instead of meter or centimer. The smoking status is also coded as 0 and 1 instead of yes no and for sex we have 7 distinct possibilities instead of 2 which we will have to fix. Finally in weight we have several individuals in the 950-1000 range. If there was one it could be a possibility, but they all seem to have a weight of 999 so I have a strong suspicion that these individuals just had missing values which has been filled in as 999. Similarily for height, some individuals have a negative height which I take to just be missing. Lets fix it!

In [207]:
dfUK.smokes = dfUK.smokes.astype(str) # Convert from integer to object just like the dtype in the polish set so I can change smoking status variable
dfUK.loc[(dfUK.smokes == 0), "smokes"] = "no" # Convert the smoking variable to the dutch standard
dfUK.loc[(dfUK.smokes == 1), "smokes"] = "yes"

In [199]:
dfUK["sex"].unique() # These are the unique values for sex column, lets dichotomize it

dfUK.loc[(dfUK.sex == "FEMALE"), "sex"] = "female"
dfUK.loc[(dfUK.sex == "Woman"), "sex"] = "female"
dfUK.loc[(dfUK.sex == "Male"), "sex"] = "male"
dfUK.loc[(dfUK.sex == "MALE"), "sex"] = "male"
dfUK.loc[(dfUK.sex == "Female"), "sex"] = "female"

In [218]:
dfUK

,id,weight,height,sex,smokes
0,UK_0001,180.8,-2.540,male,no
1,UK_0002,141.1,166.878,male,no
2,UK_0003,180.8,173.990,male,no
3,UK_0004,218.3,175.006,male,no
4,UK_0005,147.7,159.004,female,no
...,...,...,...,...,...
689,UK_0690,156.5,156.972,female,no
690,UK_0691,176.4,166.116,male,yes
691,UK_0692,205.0,183.896,male,no
692,UK_0693,130.1,167.894,female,no


In [ ]:
dfUK["height"] = dfUK["height"]*2.54     # Inches to centimeters
dfUK["weight"] = dfUK["weight"]*0.453592 # Pounds to kilogram

In [223]:
dfUK.loc[(dfUK.height <= 0), "height"] = pd.NA

In [ ]:
dfNL.dtypes
dfUS.dtypes
dfPL.dtypes
dfUK.dtypes

id            str
weight      Int64
height      Int64
sex           str
smokes        str
BMI       Float64
dtype: object